# 練習レビュー — WAV分割 + ゲイン正規化 + API登録

## やること
1. Google DriveのWAVファイルを読み込む
2. ゲインを正規化（マイクゲインが低い問題を解消）
3. 音量が明らかに低い区間で曲の切れ目を検出
4. セグメントごとに分割 + MP3変換（Web再生用）
5. 分割ファイルをGoogle Driveに保存
6. メタデータをVercel APIに登録（Webで見れるようにする）

## 初回セットアップ
1. 下の「設定」セルの `APP_URL`, `GROUP_ID` を自分の環境に合わせる
2. Google Driveのフォルダパスを確認する
3. 上から順に実行

In [ ]:
#@title 設定
from pathlib import Path

# ── あなたの環境に合わせて変更 ──
APP_URL   = "https://your-app.vercel.app"  #@param {type:"string"}
GROUP_ID  = "YOUR_LINE_GROUP_ID"            #@param {type:"string"}

# Google Drive上のパス
DRIVE_INPUT_DIR  = "/content/drive/MyDrive/_DIGITALMEAT_REC/02_練習録音・録画"  #@param {type:"string"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/_DIGITALMEAT_REC/03_分割済み"        #@param {type:"string"}

# 音声処理パラメータ
MEAN_DROP_DB   = 25.0   #@param {type:"number"} 全体平均からこのdB以上下がったら境界
MIN_LOW_SEC    = 4.0    #@param {type:"number"} 低音量がこの秒数以上続いたら境界
MIN_TRACK_SEC  = 60.0   #@param {type:"number"} これより短いセグメントはマージ
PAD_SEC        = 0.5    #@param {type:"number"} 切り出しの前後パディング
TARGET_PEAK_DB = -1.0   #@param {type:"number"} ノーマライズ目標ピーク

IN_DIR  = Path(DRIVE_INPUT_DIR)
OUT_DIR = Path(DRIVE_OUTPUT_DIR)

print(f"API: {APP_URL}")
print(f"Group: {GROUP_ID}")
print(f"Input:  {IN_DIR}")
print(f"Output: {OUT_DIR}")

In [ ]:
#@title Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

# 入力ディレクトリの確認
if IN_DIR.exists():
    wavs = sorted(IN_DIR.glob("*.wav")) + sorted(IN_DIR.glob("*.WAV"))
    print(f"\n入力フォルダ: {len(wavs)} 個のWAVファイル")
    for w in wavs:
        size_mb = w.stat().st_size / (1024*1024)
        print(f"  {w.name}  ({size_mb:.0f} MB)")
else:
    print(f"エラー: 入力フォルダが見つかりません: {IN_DIR}")

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"\n出力フォルダ: {OUT_DIR}")

In [ ]:
#@title 音声処理エンジン（split_and_normalize）
import subprocess
import re
import numpy as np
from pathlib import Path

GAIN_CAP_DB = 30.0
WINDOW_SEC  = 0.5
ANALYSIS_SR = 8000

def run_cmd(cmd):
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    return p.returncode, p.stdout, p.stderr

def get_duration(path):
    rc, out, err = run_cmd(["ffprobe", "-hide_banner", "-v", "error",
                            "-show_entries", "format=duration", "-of", "csv=p=0", str(path)])
    return float(out.strip())

def volumedetect(path):
    rc, out, err = run_cmd(["ffmpeg", "-hide_banner", "-i", str(path),
                            "-af", "volumedetect", "-f", "null", "-"])
    m_max  = re.search(r"max_volume:\s*(-?[\d.]+)\s*dB", err)
    m_mean = re.search(r"mean_volume:\s*(-?[\d.]+)\s*dB", err)
    return (float(m_max.group(1)) if m_max else None,
            float(m_mean.group(1)) if m_mean else None)

def read_audio_for_analysis(path):
    cmd = ["ffmpeg", "-hide_banner", "-v", "error", "-i", str(path),
           "-ac", "1", "-ar", str(ANALYSIS_SR), "-f", "f32le", "-acodec", "pcm_f32le", "pipe:1"]
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if p.returncode != 0:
        raise RuntimeError(f"ffmpeg failed: {p.stderr.decode()}")
    return np.frombuffer(p.stdout, dtype=np.float32), ANALYSIS_SR

def compute_rms_envelope_db(samples, sr, window_sec):
    win = int(sr * window_sec)
    n = len(samples) // win
    if n == 0: return np.array([])
    trimmed = samples[:n * win].reshape(n, win)
    rms = np.sqrt(np.mean(trimmed ** 2, axis=1))
    return 20.0 * np.log10(rms + 1e-10)

def find_low_regions(rms_db, window_sec, mean_db, drop_db, min_low_sec):
    threshold = mean_db - drop_db
    is_low = rms_db < threshold
    regions, in_region, start = [], False, 0
    for i in range(len(is_low)):
        if is_low[i] and not in_region:
            start, in_region = i, True
        elif not is_low[i] and in_region:
            if (i - start) * window_sec >= min_low_sec:
                regions.append((start * window_sec, i * window_sec))
            in_region = False
    if in_region and (len(is_low) - start) * window_sec >= min_low_sec:
        regions.append((start * window_sec, len(is_low) * window_sec))
    return regions

def regions_to_segments(low_regions, total_dur, pad_sec, min_track_sec):
    if not low_regions: return [(0.0, total_dur)]
    splits = [(r[0]+r[1])/2 for r in low_regions]
    bounds = [0.0] + splits + [total_dur]
    raw = [(bounds[i], bounds[i+1]) for i in range(len(bounds)-1)]
    merged = []
    for s, e in raw:
        if merged and (merged[-1][1] - merged[-1][0]) < min_track_sec:
            merged[-1] = (merged[-1][0], e)
        else:
            merged.append((s, e))
    if len(merged) > 1 and (merged[-1][1] - merged[-1][0]) < min_track_sec:
        merged[-2] = (merged[-2][0], merged[-1][1])
        merged.pop()
    padded = []
    for i, (s, e) in enumerate(merged):
        ps = s + pad_sec if i > 0 else s
        pe = e - pad_sec if i < len(merged)-1 else e
        if pe > ps: padded.append((ps, pe))
    return padded

def compute_gain_db(max_db):
    if max_db is None: return 0.0
    return max(0.0, min(GAIN_CAP_DB, TARGET_PEAK_DB - max_db))

def extract_and_normalize(src, out, start, end):
    dur = end - start
    rc, _, err = run_cmd(["ffmpeg", "-hide_banner",
        "-ss", f"{start:.3f}", "-t", f"{dur:.3f}", "-i", str(src),
        "-af", "volumedetect", "-f", "null", "-"])
    m = re.search(r"max_volume:\s*(-?[\d.]+)\s*dB", err)
    gain = compute_gain_db(float(m.group(1)) if m else None)
    af = f"volume={gain}dB,alimiter=limit=0.98"
    rc, _, err = run_cmd(["ffmpeg", "-hide_banner", "-y",
        "-ss", f"{start:.3f}", "-t", f"{dur:.3f}", "-i", str(src),
        "-af", af, "-acodec", "pcm_s16le", str(out)])
    return rc == 0

def wav_to_mp3(wav_path, mp3_path, bitrate="192k"):
    rc, _, err = run_cmd(["ffmpeg", "-hide_banner", "-y", "-i", str(wav_path),
                          "-codec:a", "libmp3lame", "-b:a", bitrate, str(mp3_path)])
    if rc != 0: raise RuntimeError(f"MP3 failed: {err}")

def normalize_file(src, out):
    max_db, mean_db = volumedetect(src)
    gain = compute_gain_db(max_db)
    print(f"  Volume — Max: {max_db} dB, Mean: {mean_db} dB, Gain: +{gain:.1f} dB")
    af = f"volume={gain}dB,alimiter=limit=0.98"
    rc, _, err = run_cmd(["ffmpeg", "-hide_banner", "-y", "-i", str(src),
                          "-af", af, "-acodec", "pcm_s16le", str(out)])
    return rc == 0

def fmt_time(sec):
    m, s = divmod(int(sec), 60)
    h, m = divmod(m, 60)
    return f"{h}:{m:02d}:{s:02d}" if h > 0 else f"{m}:{s:02d}"

print("音声処理エンジン 読み込み完了")

In [ ]:
#@title Google Drive API ヘルパー（ファイルID取得用）
from google.colab import auth
from googleapiclient.discovery import build

auth.authenticate_user()
drive_service = build('drive', 'v3')

def get_drive_file_id(local_path: str) -> str:
    """Google DriveにマウントされたファイルのDrive File IDを取得する。
    
    /content/drive/MyDrive/... のパスからファイル名で検索して
    Drive APIのfile IDを返す。
    """
    p = Path(local_path)
    name = p.name
    
    # ファイル名で検索
    query = f"name = '{name}' and trashed = false"
    results = drive_service.files().list(
        q=query, fields="files(id, name, parents)", pageSize=10
    ).execute()
    files = results.get('files', [])
    
    if not files:
        print(f"  Warning: Drive file ID not found for {name}")
        return None
    
    # 複数ヒットした場合は最初のものを使う
    file_id = files[0]['id']
    if len(files) > 1:
        print(f"  Note: {len(files)} files named '{name}' found, using first")
    
    return file_id

def share_file_readonly(file_id: str):
    """ファイルをリンクで閲覧可能にする（Web再生に必要）。"""
    try:
        drive_service.permissions().create(
            fileId=file_id,
            body={'type': 'anyone', 'role': 'reader'},
        ).execute()
    except Exception as e:
        # Already shared or permission error — not critical
        print(f"  Share warning: {e}")

print("Google Drive API 認証完了")

In [ ]:
#@title Vercel API ヘルパー
import requests

def api_post(path, data):
    resp = requests.post(f"{APP_URL}{path}", json=data)
    resp.raise_for_status()
    return resp.json()

def api_get(path):
    resp = requests.get(f"{APP_URL}{path}")
    resp.raise_for_status()
    return resp.json()

# 接続テスト
try:
    health = requests.get(f"{APP_URL}/health").json()
    print(f"API接続OK: {health}")
except Exception as e:
    print(f"API接続エラー: {e}")
    print("APP_URL が正しいか確認してください")

In [ ]:
#@title メイン処理: WAV分割 → ゲイン正規化 → MP3変換 → API登録
import os
from datetime import datetime

wavs = sorted(IN_DIR.glob("*.wav")) + sorted(IN_DIR.glob("*.WAV"))
wavs = list({p.resolve(): p for p in wavs}.values())  # dedupe

print(f"処理対象: {len(wavs)} ファイル")
print(f"設定: drop={MEAN_DROP_DB}dB, min_low={MIN_LOW_SEC}s, min_track={MIN_TRACK_SEC}s")
print()

for wav in wavs:
    stem = wav.stem
    file_out_dir = OUT_DIR / stem
    file_out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"File: {wav.name}")

    # --- 1. 音声処理 ---
    total_dur = get_duration(wav)
    print(f"  Duration: {fmt_time(total_dur)} ({total_dur/60:.1f} min)")

    # Normalize
    norm_path = file_out_dir / f"{stem}_normalized.wav"
    print("  Normalizing...")
    normalize_file(wav, norm_path)

    # MP3 of full session
    session_mp3 = file_out_dir / f"{stem}.mp3"
    print("  Creating session MP3...")
    wav_to_mp3(norm_path, session_mp3)

    # Analyze envelope
    print("  Analyzing volume envelope...")
    samples, sr = read_audio_for_analysis(norm_path)
    rms_db = compute_rms_envelope_db(samples, sr, WINDOW_SEC)
    del samples

    overall_mean = float(np.mean(rms_db))
    print(f"  Envelope mean: {overall_mean:.1f} dB")
    print(f"  Boundary: < {overall_mean - MEAN_DROP_DB:.1f} dB for {MIN_LOW_SEC}s+")

    low_regions = find_low_regions(rms_db, WINDOW_SEC, overall_mean, MEAN_DROP_DB, MIN_LOW_SEC)
    del rms_db

    if low_regions:
        print(f"  Found {len(low_regions)} boundary(s):")
        for i, (s, e) in enumerate(low_regions):
            print(f"    [{i+1}] {fmt_time(s)} ~ {fmt_time(e)} ({e-s:.1f}s)")
    else:
        print("  No boundaries detected — single track")

    segments = regions_to_segments(low_regions, total_dur, PAD_SEC, MIN_TRACK_SEC)
    print(f"  → {len(segments)} segment(s)")

    # Extract segments
    seg_files = []
    for i, (s, e) in enumerate(segments):
        num = i + 1
        seg_wav = file_out_dir / f"{stem}_track{num:02d}.wav"
        seg_mp3 = file_out_dir / f"{stem}_track{num:02d}.mp3"
        print(f"  [Track {num:02d}] {fmt_time(s)} ~ {fmt_time(e)}")

        if extract_and_normalize(norm_path, seg_wav, s, e):
            wav_to_mp3(seg_wav, seg_mp3)
            seg_files.append({"num": num, "start": s, "end": e,
                              "wav": seg_wav, "mp3": seg_mp3})

    # Cleanup normalized intermediate
    try: norm_path.unlink()
    except: pass

    # --- 2. Drive File ID 取得 ---
    print("\n  Getting Drive file IDs...")
    session_mp3_id = get_drive_file_id(str(session_mp3))
    original_wav_id = get_drive_file_id(str(wav))

    if session_mp3_id:
        share_file_readonly(session_mp3_id)

    for sf in seg_files:
        sf["mp3_id"] = get_drive_file_id(str(sf["mp3"]))
        sf["wav_id"] = get_drive_file_id(str(sf["wav"]))
        if sf["mp3_id"]:
            share_file_readonly(sf["mp3_id"])

    # --- 3. API登録 ---
    print("\n  Registering with API...")

    # Try to parse date from filename (e.g., "2026-03-10_スタジオ練習")
    recorded_at = None
    date_match = re.search(r"(\d{4}[-_]\d{2}[-_]\d{2})", stem)
    if date_match:
        try:
            recorded_at = datetime.strptime(
                date_match.group(1).replace('_', '-'), "%Y-%m-%d"
            ).isoformat()
        except ValueError:
            pass

    session_data = api_post("/api/sessions", {
        "group_id": GROUP_ID,
        "title": stem,
        "recorded_at": recorded_at,
        "duration_sec": total_dur,
        "drive_file_id": original_wav_id,
        "drive_mp3_id": session_mp3_id,
        "status": "processed",
    })
    session_id = session_data["id"]
    print(f"  Session registered: id={session_id}")

    # Register segments
    seg_payload = [
        {
            "track_number": sf["num"],
            "start_sec": sf["start"],
            "end_sec": sf["end"],
            "drive_file_id": sf.get("wav_id"),
            "drive_mp3_id": sf.get("mp3_id"),
            "auto_detected": True,
        }
        for sf in seg_files
    ]

    if seg_payload:
        result = api_post(f"/api/sessions/{session_id}/segments", seg_payload)
        print(f"  Segments registered: {len(result)} segments")

    print(f"\n  Web UI: {APP_URL}/practice/view?group_id={GROUP_ID}")

print(f"\n{'='*60}")
print("全ファイル処理完了!")
print(f"\nWeb UIで確認: {APP_URL}/practice/view?group_id={GROUP_ID}")

In [ ]:
#@title [オプション] 切り抜きリクエストの処理
#@markdown Web UIから送信された切り抜きリクエストを処理します。
#@markdown このセルを定期的に実行するか、ループで回してください。

import time

def process_clip_requests():
    """Pending clip requests を処理する。"""
    clips = api_get("/api/clips/pending")
    if not clips:
        print("Pending clip requests なし")
        return

    print(f"{len(clips)} 件の切り抜きリクエスト")

    for clip in clips:
        clip_id = clip["id"]
        session_id = clip["session_id"]
        start = clip["start_sec"]
        end = clip["end_sec"]
        title = clip.get("title") or f"clip_{clip_id}"
        drive_file_id = clip.get("drive_file_id")

        print(f"\n  Clip #{clip_id}: {fmt_time(start)} ~ {fmt_time(end)}")

        if not drive_file_id:
            print("    SKIP: no drive_file_id")
            continue

        # セッション情報を取得
        session_info = api_get(f"/api/sessions/{session_id}")
        session_title = session_info["title"]

        # 元のWAVファイルを見つける
        # Drive上のファイルはマウントされているので、ファイル名で検索
        src_candidates = list(IN_DIR.glob(f"{session_title}*")) + \
                         list(OUT_DIR.glob(f"**/{session_title}_normalized.wav"))
        if not src_candidates:
            print(f"    SKIP: source file not found for '{session_title}'")
            continue

        src = src_candidates[0]
        clip_out_dir = OUT_DIR / "_clips"
        clip_out_dir.mkdir(parents=True, exist_ok=True)

        safe_title = re.sub(r'[^\w\-]', '_', title)
        clip_wav = clip_out_dir / f"{safe_title}.wav"
        clip_mp3 = clip_out_dir / f"{safe_title}.mp3"

        try:
            extract_and_normalize(src, clip_wav, start, end)
            wav_to_mp3(clip_wav, clip_mp3)

            clip_mp3_id = get_drive_file_id(str(clip_mp3))
            if clip_mp3_id:
                share_file_readonly(clip_mp3_id)

            # Update status
            requests.patch(f"{APP_URL}/api/clips/{clip_id}", json={
                "status": "done",
                "drive_file_id": clip_mp3_id,
            })
            print(f"    Done: {clip_mp3.name}")

        except Exception as e:
            print(f"    Error: {e}")
            requests.patch(f"{APP_URL}/api/clips/{clip_id}", json={"status": "error"})

# 1回実行
process_clip_requests()

# ループで回す場合は以下のコメントを外す:
# while True:
#     process_clip_requests()
#     time.sleep(60)  # 1分ごとにチェック